<a href="https://colab.research.google.com/github/ashioyajotham/Daily-ML/blob/main/gemma4_agent_coding_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Agent vs Agent: Gemma 4 Local vs Antigravity
### *How far can open models go — and what are you actually trading?*

---

**Demo task:** Ask both agents to build a FastAPI Todo REST API with a full pytest test suite.

**Setup:** Gemma 4 E4B (4-bit quantized) via HuggingFace Transformers + BitsAndBytes on a T4 GPU.

**Scorecard metrics:**
| Metric | Gemma 4 (Local) | Antigravity (Cloud) |
|---|---|---|
| Steps to working code | ⏳ | ⏳ |
| Human interventions | ⏳ | ⏳ |
| Time to green tests | ⏳ | ⏳ |
| Errors self-corrected | ⏳ | ⏳ |

> **Hardware:** Google Colab T4 (16GB VRAM) · **Model:** `google/gemma-4-e4b-it` · **Quant:** 4-bit NF4

## ⚙️ Step 1 — Install Dependencies

> **Important:** Gemma 4 requires `transformers` installed from the GitHub source repo.
> Colab's pre-installed stable version (`~4.40`) doesn't know the `gemma4` architecture yet.
>
> **Flow:**
> 1. Run this cell — it installs from source and verifies registration
> 2. If you see the ✅ messages, proceed to Step 2
> 3. If you get a `KeyError: gemma4` later — go to **Runtime → Restart session** and re-run from here

In [1]:
# Gemma 4 needs transformers from source — Colab's bundled version
# doesn't register the 'gemma4' model_type and will throw a KeyError.
# Run this cell once, then wait for the automatic runtime restart.
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q 'bitsandbytes>=0.45.0' 'accelerate>=0.30.0'
!pip install -q fastapi uvicorn httpx pytest pytest-asyncio anyio

# Verify gemma4 is now registered before we proceed
from transformers import CONFIG_MAPPING
assert 'gemma4' in CONFIG_MAPPING, "gemma4 still not registered — check install"

import transformers
print(f"✅ transformers {transformers.__version__}")
print("✅ gemma4 architecture registered")
print("✅ All deps installed — proceed to Step 2")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
✅ transformers 5.6.0.dev0
✅ gemma4 architecture registered
✅ All deps installed — proceed to Step 2


## 🔐 Step 2 — Authenticate with HuggingFace

> Gemma 4 is gated — you need to accept the license at https://huggingface.co/google/gemma-4-e4b-it and add your HF token below (or use Colab Secrets).

In [2]:
from huggingface_hub import login
from google.colab import userdata

# Use Colab Secrets (recommended) — add HF_TOKEN in the key icon on the left sidebar
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Logged in via Colab Secrets")
except Exception:
    # Fallback: paste token directly (not recommended for sharing)
    login()  # will prompt interactively

✅ Logged in via Colab Secrets


## 🧠 Step 3 — Load Gemma 4 E4B in 4-bit

**Why E4B?** It's the largest Gemma 4 variant that fits comfortably in a T4's 16GB VRAM at 4-bit precision.
The "E" stands for *effective* parameters — it uses Per-Layer Embeddings (PLE) to punch above its weight class.

In [3]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-e4b-it"

from transformers import CONFIG_MAPPING
if 'gemma4' not in CONFIG_MAPPING:
    raise RuntimeError(
        "gemma4 not in CONFIG_MAPPING.\n"
        "Go to Runtime → Restart session, then re-run from Step 1."
    )

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"⏳ Loading {MODEL_ID}...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,          # torch_dtype is deprecated in new transformers
    attn_implementation="eager",   # safe fallback for Gemma4's hybrid attention
)

load_time = time.time() - t0
vram_used = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9

# hf_device_map only exists on standard CausalLM — Gemma4ForConditionalGeneration
# is a multimodal class and doesn't set it. Fall back gracefully.
device_info = getattr(model, 'hf_device_map', None) or str(next(model.parameters()).device)

print(f"✅ Model loaded in {load_time:.1f}s")
print(f"📦 VRAM used: {vram_used:.2f} GB / {vram_total:.0f} GB")
print(f"🔧 Device: {device_info}")
print(f"🏗️  Model class: {type(model).__name__}")

⏳ Loading google/gemma-4-e4b-it...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

✅ Model loaded in 207.0s
📦 VRAM used: 9.30 GB / 16 GB
🔧 Device: cuda:0
🏗️  Model class: Gemma4ForConditionalGeneration


## 🔧 Step 4 — The Agent Harness

A lightweight harness that:
- Wraps generation with timing + token counting
- Extracts code blocks from the response
- Writes files to disk so we can actually *run* the generated code
- Tracks intervention count (how many times we had to re-prompt)

In [4]:
import re
import os
import subprocess
from dataclasses import dataclass
from typing import Optional
from IPython.display import display, Markdown

@dataclass
class RunMetrics:
    """Scorecard for the demo comparison."""
    model_name: str
    steps: int = 0
    interventions: int = 0
    total_time_s: float = 0.0
    errors_self_corrected: int = 0
    tests_passed: Optional[bool] = None
    tokens_generated: int = 0

    def display(self):
        status = "✅ PASS" if self.tests_passed else ("❌ FAIL" if self.tests_passed is False else "⏳ N/A")
        display(Markdown(f"""
### 📊 Scorecard — `{self.model_name}`
| Metric | Result |
|---|---|
| Steps to working code | {self.steps} |
| Human interventions | {self.interventions} |
| Time to completion | {self.total_time_s:.1f}s |
| Errors self-corrected | {self.errors_self_corrected} |
| Tokens generated | {self.tokens_generated:,} |
| Test suite result | {status} |
"""))


def generate(
    prompt: str,
    system: str = "You are an expert Python software engineer. Write complete, working code.",
    max_new_tokens: int = 2048,
    temperature: float = 0.2,
) -> tuple[str, float, int]:
    """Run a single generation pass. Returns (response_text, elapsed_s, n_tokens)."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]

    # apply_chat_template may return a raw tensor OR a BatchEncoding dict
    # depending on transformers version + model type. Normalise explicitly.
    raw = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    )

    if hasattr(raw, "input_ids"):
        # BatchEncoding — extract tensors and move to device
        input_ids      = raw.input_ids.to(model.device)
        attention_mask = raw.attention_mask.to(model.device)
        gen_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
    else:
        # Plain tensor
        input_ids  = raw.to(model.device)
        gen_kwargs = {"input_ids": input_ids}

    n_input = input_ids.shape[-1]

    t0 = time.time()
    with torch.inference_mode():
        output_ids = model.generate(
            **gen_kwargs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0

    new_tokens = output_ids[0][n_input:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response, elapsed, len(new_tokens)


def extract_code_blocks(text: str) -> dict[str, str]:
    """
    Extract fenced code blocks. Infers filename from first-line comment
    e.g. '# main.py' — otherwise names them block_1.py, block_2.py etc.
    """
    pattern = r"```(?:\w+)?\n(.*?)```"
    blocks = {}
    for i, m in enumerate(re.finditer(pattern, text, re.DOTALL)):
        code = m.group(1).strip()
        first_line = code.split("\n")[0]
        fname_match = re.match(r"#\s*([\w./]+\.py)", first_line)
        key = fname_match.group(1) if fname_match else f"block_{i+1}.py"
        blocks[key] = code
    return blocks


def write_files(blocks: dict[str, str], output_dir: str = "/tmp/gemma_output") -> list[str]:
    """Write extracted code blocks to disk. Auto-renames test files if needed."""
    os.makedirs(output_dir, exist_ok=True)
    written = []
    for fname, code in blocks.items():
        base = os.path.basename(fname)

        # Pytest only discovers test_*.py — rename common variants automatically
        test_aliases = {"tests.py", "api_test.py", "test.py", "tests_main.py"}
        has_test_fn  = "def test_" in code
        not_named    = not base.startswith("test_")

        if has_test_fn and not_named:
            # Check if we already have a test file being written this round
            existing_test = next((b for b in blocks if os.path.basename(b).startswith("test_")), None)
            if existing_test is None or base in test_aliases:
                base = "test_main.py"
                print(f"  ⚠️  Renamed '{fname}' → 'test_main.py' (pytest discovery fix)")

        path = os.path.join(output_dir, base)
        with open(path, "w") as f:
            f.write(code)
        written.append(path)
        print(f"  📄 Written: {path}")
    return written


def run_tests(test_dir: str = "/tmp/gemma_output") -> tuple[bool, str]:
    """Run pytest. Returns (passed, output). Diagnoses empty collection."""
    files = os.listdir(test_dir) if os.path.isdir(test_dir) else []
    test_files = [f for f in files if f.startswith("test_") and f.endswith(".py")]

    if not test_files:
        # Diagnose: list what IS there so we can fix it
        diag = f"⚠️  pytest collected 0 items — no test_*.py files found.\n"
        diag += f"   Files in {test_dir}: {files}\n"
        for f in files:
            path = os.path.join(test_dir, f)
            snippet = open(path).read()[:300]
            has_tests = "def test_" in snippet
            diag += f"   {f}: has 'def test_'={has_tests}\n"
        return False, diag

    result = subprocess.run(
        ["python", "-m", "pytest", test_dir, "-v", "--tb=short", "--no-header"],
        capture_output=True, text=True, cwd=test_dir
    )
    return result.returncode == 0, result.stdout + result.stderr


print("✅ Harness ready")

✅ Harness ready


## 🎯 Step 5 — The Task Prompt

This is the **exact same prompt** sent to both Gemma 4 and Antigravity. No favoritism.

In [5]:
TASK_PROMPT = """
Build a complete, working Python REST API with the following requirements:

**API (main.py):**
- Framework: FastAPI
- Resource: Todo items with fields: id (int), title (str), completed (bool), created_at (datetime)
- Endpoints:
  - GET  /todos        — list all todos
  - POST /todos        — create a new todo
  - GET  /todos/{id}   — get single todo
  - PUT  /todos/{id}   — update a todo
  - DELETE /todos/{id} — delete a todo
- In-memory storage (a plain Python dict is fine, no database needed)
- Proper HTTP status codes (201, 404, 200, 204)
- Pydantic models for request/response validation

**Tests (test_main.py):**
- Framework: pytest + httpx (use AsyncClient with app)
- Test every endpoint: happy path + edge cases (missing item, duplicate, etc.)
- At least 8 test functions

Output both files in full. No placeholders. The code must run as-is.
"""

display(Markdown(f"**Task prompt ({len(TASK_PROMPT)} chars):**\n```\n{TASK_PROMPT.strip()}\n```"))

**Task prompt (858 chars):**
```
Build a complete, working Python REST API with the following requirements:

**API (main.py):**
- Framework: FastAPI
- Resource: Todo items with fields: id (int), title (str), completed (bool), created_at (datetime)
- Endpoints:
  - GET  /todos        — list all todos
  - POST /todos        — create a new todo
  - GET  /todos/{id}   — get single todo
  - PUT  /todos/{id}   — update a todo
  - DELETE /todos/{id} — delete a todo
- In-memory storage (a plain Python dict is fine, no database needed)
- Proper HTTP status codes (201, 404, 200, 204)
- Pydantic models for request/response validation

**Tests (test_main.py):**
- Framework: pytest + httpx (use AsyncClient with app)
- Test every endpoint: happy path + edge cases (missing item, duplicate, etc.)
- At least 8 test functions

Output both files in full. No placeholders. The code must run as-is.
```

## 🚀 Step 6 — Run Gemma 4 on the Task

⏱️ *This will take ~2–4 minutes on T4 at 4-bit. Watch the token count.*

In [ ]:
metrics = RunMetrics(model_name="Gemma 4 E4B (4-bit, T4)")

print("🤖 Sending task to Gemma 4 E4B...\n")
print("─" * 60)

# ── Step 1: Initial generation ──────────────────────────────
t_total_start = time.time()
response, elapsed, n_tokens = generate(TASK_PROMPT)
metrics.steps += 1
metrics.tokens_generated += n_tokens

print(f"⚡ Generated {n_tokens:,} tokens in {elapsed:.1f}s ({n_tokens/elapsed:.0f} tok/s)")
print("\n" + "─" * 60)
print("📝 Raw response preview (first 1000 chars):")
print(response[:1000])
print("..." if len(response) > 1000 else "")

🤖 Sending task to Gemma 4 E4B...

────────────────────────────────────────────────────────────


In [ ]:
# ── Step 2: Extract and write code blocks ───────────────────
print("\n🔍 Extracting code blocks...")
blocks = extract_code_blocks(response)
print(f"   Found {len(blocks)} block(s): {list(blocks.keys())}")

print("\n💾 Writing files to disk...")
written = write_files(blocks)

# If model didn't name files correctly, we can map manually
# Uncomment and adjust if needed:
# import shutil
# shutil.copy("/tmp/gemma_output/block_1.py", "/tmp/gemma_output/main.py")
# shutil.copy("/tmp/gemma_output/block_2.py", "/tmp/gemma_output/test_main.py")

In [ ]:
# ── Step 3: Run the tests ────────────────────────────────────
print("\n🧪 Running pytest...\n")
passed, test_output = run_tests()

metrics.tests_passed = passed
metrics.total_time_s = time.time() - t_total_start

print(test_output)
print("─" * 60)
print(f"{'✅ Tests PASSED' if passed else '❌ Tests FAILED — intervention needed'}")

In [ ]:
# ── Step 4 (conditional): Self-correction loop ───────────────
# If tests failed, give Gemma the error output and ask it to fix

MAX_RETRIES = 2

if not passed:
    for attempt in range(MAX_RETRIES):
        metrics.interventions += 1
        print(f"\n🔄 Self-correction attempt {attempt + 1}/{MAX_RETRIES}...")

        fix_prompt = f"""
The code you wrote has failing tests. Here is the pytest output:

```
{test_output[-2000:]}
```

Here is the current main.py:
```python
{open('/tmp/gemma_output/main.py').read() if os.path.exists('/tmp/gemma_output/main.py') else 'FILE NOT FOUND'}
```

Here is the current test_main.py:
```python
{open('/tmp/gemma_output/test_main.py').read() if os.path.exists('/tmp/gemma_output/test_main.py') else 'FILE NOT FOUND'}
```

Fix ALL issues. Output complete corrected versions of both files.
"""
        response, elapsed, n_tokens = generate(fix_prompt, max_new_tokens=3000)
        metrics.steps += 1
        metrics.tokens_generated += n_tokens
        print(f"   ⚡ {n_tokens:,} tokens in {elapsed:.1f}s")

        blocks = extract_code_blocks(response)
        write_files(blocks)

        passed, test_output = run_tests()
        if passed:
            metrics.errors_self_corrected += 1
            metrics.tests_passed = True
            print(f"   ✅ Fixed on attempt {attempt + 1}!")
            break
        else:
            print(f"   ❌ Still failing. {'Giving up.' if attempt == MAX_RETRIES - 1 else 'Retrying...'}")

metrics.total_time_s = time.time() - t_total_start
print(f"\n⏱️ Total elapsed: {metrics.total_time_s:.1f}s")

## 📊 Step 7 — Scorecard

In [ ]:
metrics.display()

# Print the final generated files for inspection
print("\n" + "═" * 60)
print("📄 FINAL main.py:")
print("═" * 60)
if os.path.exists("/tmp/gemma_output/main.py"):
    print(open("/tmp/gemma_output/main.py").read())

print("\n" + "═" * 60)
print("📄 FINAL test_main.py:")
print("═" * 60)
if os.path.exists("/tmp/gemma_output/test_main.py"):
    print(open("/tmp/gemma_output/test_main.py").read())

## 🚁 Step 8 — Antigravity Side (Live Demo)

This section is performed **live** in Google Antigravity during the talk.

### Exact prompt to paste into Antigravity Agent Manager:

```
Build a complete, working Python REST API with the following requirements:

API (main.py):
- Framework: FastAPI
- Resource: Todo items with fields: id (int), title (str), completed (bool), created_at (datetime)
- Endpoints:
  - GET  /todos        — list all todos
  - POST /todos        — create a new todo
  - GET  /todos/{id}   — get single todo
  - PUT  /todos/{id}   — update a todo
  - DELETE /todos/{id} — delete a todo
- In-memory storage (a plain Python dict is fine, no database needed)
- Proper HTTP status codes (201, 404, 200, 204)
- Pydantic models for request/response validation

Tests (test_main.py):
- Framework: pytest + httpx (use AsyncClient with app)
- Test every endpoint: happy path + edge cases (missing item, duplicate, etc.)
- At least 8 test functions

Run the tests after writing the code and fix any failures automatically.
```

### What to watch for (narrate to audience):
1. **Plan Artifact** — does it create a plan before coding?
2. **Multi-file creation** — does it create `main.py` and `test_main.py` as separate steps?
3. **Terminal access** — does it run `pytest` on its own?
4. **Self-correction** — does it fix failures without being asked?
5. **Verification** — does it confirm green tests before declaring done?

### Fill in the scorecard live:
| Metric | Gemma 4 (this notebook) | Antigravity (live) |
|---|---|---|
| Steps | `metrics.steps` | ??? |
| Human interventions | `metrics.interventions` | ??? |
| Time | `metrics.total_time_s` | ??? |
| Self-corrections | `metrics.errors_self_corrected` | ??? |
| Tests passed | `metrics.tests_passed` | ??? |

## 🧭 Step 9 — Analysis & The Real Question

After both runs, fill in and run this cell to surface the tradeoff framing.

In [ ]:
# Fill these in from the Antigravity live run
antigravity_metrics = RunMetrics(
    model_name="Antigravity (Gemini 3.1 Pro, Cloud)",
    steps=0,           # fill in
    interventions=0,   # fill in
    total_time_s=0.0,  # fill in
    errors_self_corrected=0,  # fill in
    tests_passed=None, # fill in: True/False
    tokens_generated=0,
)

# Side-by-side summary
display(Markdown("""
## 🏆 Final Comparison
"""))

metrics.display()
antigravity_metrics.display()

display(Markdown("""
---
## 🧭 The Real Question

| Dimension | Antigravity wins | Gemma 4 Local wins |
|---|---|---|
| **Raw capability** | ✅ | |
| **Agentic orchestration** | ✅ | |
| **Privacy / data sovereignty** | | ✅ |
| **Cost (zero quota)** | | ✅ |
| **Offline / low-bandwidth** | | ✅ |
| **Customisability / fine-tuning** | | ✅ |

> *"The question isn't which is better. It's which is right for your context."*
>
> For an African developer building sensitive apps, working in areas with unreliable internet,
> or hitting quota limits — Gemma 4 local isn't a compromise. It's a choice.
"""))

---
**Built for the Google Antigravity Buildathon · Nairobi 2026**  
Victor Ashioya (Jotham) · [@ashioyajotham](https://github.com/ashioyajotham) · [ashioyajotham.github.io](https://ashioyajotham.github.io)